# Multi-Layer Perceptron (MLP) for Regression

## 1. Introduction

Neural networks are among the most important machine learning models used today.
They are inspired by the functioning of the human brain and are designed to learn complex relationships between data.

Among the simplest neural network architectures, the **Multi-Layer Perceptron (MLP)** is widely used for:

* ✅ regression tasks,
* ✅ classification tasks,
* ✅ prediction systems,
* ✅ pattern recognition.

In this notebook, we study the implementation of an MLP for regression tasks using the custom implementation provided in the repository. We also compare this approach with implementations available in scikit-learn.

## 2. Problem Statement

Traditional machine learning algorithms such as:

* Linear Regression,
* K-Nearest Neighbors (KNN),
* Decision Trees,

can struggle when the relationship between the input data and the target values becomes **highly non-linear**.

Neural networks address this limitation by learning complex mathematical representations directly from the data.

The goal of this notebook is therefore to understand:

1. how a neural network works,
2. how it learns,
3. and how it can be implemented from scratch.

## 3. Multi-Layer Perceptron (MLP)

A Multi-Layer Perceptron is a **feedforward neural network** composed of:

* an **input layer**,
* one or more **hidden layers**,
* an **output layer**.

The model learns by adjusting **weights** and **biases** during training.

The global behavior of the network can be represented by:

$$
y = f(x; \theta)
$$

Where:
* $x$ represents the input data,
* $\theta$ represents the trainable parameters of the network (weights and biases),
* $f$ is the learned function.

## 4. Architecture of the Network

The implemented model contains several important components.

### 4.1 Input Layer

The input layer receives the features of the dataset.

**Examples of features:**
* house size,
* number of rooms,
* temperature,
* financial indicators.

### 4.2 Hidden Layers

The hidden layers perform transformations on the data.

Their sizes are defined using `hidden_layer_sizes = (64, 32)` where:
* First hidden layer: 64 neurons
* Second hidden layer: 32 neurons

These layers allow the network to learn complex patterns and non-linear relationships.

### 4.3 Output Layer

Since this implementation focuses on **regression tasks**, the output layer uses a **linear activation**.

The network predicts continuous numerical values (e.g., price, temperature, probability).

## 5. Activation Functions

Activation functions introduce **non-linearity** into the network.

Without them, the network would simply be a linear regression model, regardless of the number of layers.

Several activation functions are implemented in the project:

### 5.1 Sigmoid

The sigmoid function transforms values into the interval $[0, 1]$.

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$

**Use case:** Binary classification outputs, probability estimation.

### 5.2 ReLU (Rectified Linear Unit)

ReLU is one of the most commonly used activation functions.

$$
f(x) = \max(0, x)
$$

**Advantages:** Helps neural networks train faster and reduces vanishing gradient problems.

### 5.3 Tanh (Hyperbolic Tangent)

Outputs values between $-1$ and $1$.

$$
\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}
$$

**Use case:** When outputs need to be centered around zero.

### 5.4 Leaky ReLU

Leaky ReLU is an improved version of ReLU that prevents neurons from becoming inactive ("dying ReLU" problem).

$$
f(x) = \begin{cases} x & \text{if } x > 0 \\ 0.01x & \text{otherwise} \end{cases}
$$

## 6. Forward Propagation

Forward propagation is the process through which data moves across the network.

For each neuron:

$$
a = \sigma(Wx + b)
$$

Where:
* $W$ represents the **weights** (connections between neurons),
* $b$ represents the **biases** (offset),
* $\sigma$ is the **activation function**.

The forward propagation process is implemented in the `_forward_pass` method of our MLPRegressor class.

## 7. Loss Function

The model uses the **Mean Squared Error (MSE)** loss function with **L2 regularization**.

$$
\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2
$$

With L2 regularization:

$$
\text{Loss} = \text{MSE} + \frac{\alpha}{2m}\sum_{j} w_j^2
$$

Where:
* $y_i$ are the true values,
* $\hat{y}_i$ are the predicted values,
* $\alpha$ is the regularization parameter (penalizes large weights to prevent overfitting).

**The lower the loss:** the better the model performs.

## 8. Backpropagation

Backpropagation is the **learning mechanism** of the neural network.

It works as follows:

1. Compute the **gradient of the loss** with respect to each weight using the chain rule
2. Update the weights in the **opposite direction** of the gradient (gradient descent)

$$
w_{\text{new}} = w_{\text{old}} - \eta \frac{\partial \text{Loss}}{\partial w}
$$

Where $\eta$ is the **learning rate**.

Backpropagation is implemented in the `_backward_pass` method of our MLPRegressor.

## 9. Optimizers

The model supports multiple **optimizers** to update weights efficiently:

### 9.1 SGD (Stochastic Gradient Descent)

The simplest optimizer. Updates weights using the gradient of the loss.

### 9.2 Momentum

Adds momentum to gradient descent to accelerate convergence and reduce oscillations.

### 9.3 RMSProp

Adapts the learning rate for each parameter by dividing by a moving average of squared gradients.

### 9.4 Adam (Adaptive Moment Estimation)

Combines Momentum and RMSProp. This is usually the **best performing** optimizer for most problems.

## 10. Regularization and Early Stopping

To prevent **overfitting**, the model implements:

* **L2 Regularization** ($\alpha$ parameter) – penalizes large weights
* **Early Stopping** – stops training when validation loss stops improving

Early stopping checks if the validation loss hasn't improved for `n_iter_no_change` epochs.

## 11. Importing Libraries

In [ ]:
from typing import List, Tuple, Optional
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor as SklearnMLP

# For better plots
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 12. Custom MLPRegressor Implementation

Below is the complete implementation of our Multi-Layer Perceptron for regression tasks.

In [ ]:
class MLPRegressor:
    """
    Multi-Layer Perceptron for regression tasks with various activation functions and optimizers
    
    This implementation follows the architecture described in sections 3-10.
    """
    
    def __init__(
        self,
        hidden_layer_sizes: Tuple[int, ...] = (100,),
        activation: str = "relu",
        solver: str = "sgd",
        alpha: float = 0.0001,
        batch_size: int = 32,
        learning_rate: float = 0.001,
        max_iter: int = 200,
        shuffle: bool = True,
        random_state: Optional[int] = None,
        beta1: float = 0.9,
        beta2: float = 0.999,
        epsilon: float = 1e-8,
        momentum: float = 0.9,
        tol: float = 1e-4,
        early_stopping: bool = False,
        validation_fraction: float = 0.1,
        n_iter_no_change: int = 10
    ):
        """Initialize an MLP regressor"""
        self.hidden_layer_sizes = hidden_layer_sizes
        self.activation = activation
        self.solver = solver
        self.alpha = alpha
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.shuffle = shuffle
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.momentum = momentum
        self.tol = tol
        self.early_stopping = early_stopping
        self.validation_fraction = validation_fraction
        self.n_iter_no_change = n_iter_no_change
        
        if random_state is not None:
            np.random.seed(random_state)
        
        # Activation functions (see section 5)
        self.activation_functions = {
            'sigmoid': self._sigmoid,
            'relu': self._relu,
            'tanh': self._tanh,
            'leaky_relu': self._leaky_relu,
        }
        
        self.activation_derivatives = {
            'sigmoid': self._sigmoid_derivative,
            'relu': self._relu_derivative,
            'tanh': self._tanh_derivative,
            'leaky_relu': self._leaky_relu_derivative,
        }
        
        if activation not in self.activation_functions:
            raise ValueError(f"Activation '{activation}' not recognized.")
        
        self.activation_func = self.activation_functions[activation]
        self.activation_derivative = self.activation_derivatives[activation]
        
        # Network parameters
        self.weights = []
        self.biases = []
        self.n_layers = None
        self.n_outputs = None
        
        # Optimizer states
        self.velocity_weights = []  # For Momentum
        self.velocity_biases = []
        self.m_weights = []  # For Adam
        self.m_biases = []
        self.v_weights = []  # For Adam
        self.v_biases = []
        self.t = 1
        
        self.loss_history = []
        self.val_loss_history = []
        self.best_loss = np.inf
        self.no_improvement_count = 0
        self.trained = False
    
    def _initialize_weights(self, n_features: int, n_outputs: int) -> None:
        """Initialize weights using Xavier/Glorot initialization"""
        layer_sizes = [n_features] + list(self.hidden_layer_sizes) + [n_outputs]
        self.n_layers = len(layer_sizes) - 1
        self.n_outputs = n_outputs
        
        # Reset lists
        self.weights = []
        self.biases = []
        self.velocity_weights = []
        self.velocity_biases = []
        self.m_weights = []
        self.m_biases = []
        self.v_weights = []
        self.v_biases = []
        
        # Xavier/Glorot initialization
        for i in range(self.n_layers):
            limit = np.sqrt(6 / (layer_sizes[i] + layer_sizes[i + 1]))
            self.weights.append(np.random.uniform(-limit, limit, (layer_sizes[i], layer_sizes[i + 1])))
            self.biases.append(np.zeros(layer_sizes[i + 1]))
            
            # Optimizer states
            self.velocity_weights.append(np.zeros_like(self.weights[-1]))
            self.velocity_biases.append(np.zeros_like(self.biases[-1]))
            self.m_weights.append(np.zeros_like(self.weights[-1]))
            self.m_biases.append(np.zeros_like(self.biases[-1]))
            self.v_weights.append(np.zeros_like(self.weights[-1]))
            self.v_biases.append(np.zeros_like(self.biases[-1]))
    
    # ==================== Activation Functions ====================
    
    def _leaky_relu(self, x: np.ndarray) -> np.ndarray:
        """Leaky ReLU activation function (section 5.4)"""
        return np.where(x > 0, x, 0.01 * x)
    
    def _leaky_relu_derivative(self, x: np.ndarray) -> np.ndarray:
        """Derivative of Leaky ReLU"""
        return np.where(x > 0, 1, 0.01)
    
    def _sigmoid(self, x: np.ndarray) -> np.ndarray:
        """Sigmoid activation function (section 5.1)"""
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def _sigmoid_derivative(self, x: np.ndarray) -> np.ndarray:
        """Derivative of sigmoid"""
        sigmoid_x = self._sigmoid(x)
        return sigmoid_x * (1 - sigmoid_x)
    
    def _relu(self, x: np.ndarray) -> np.ndarray:
        """ReLU activation function (section 5.2)"""
        return np.maximum(0, x)
    
    def _relu_derivative(self, x: np.ndarray) -> np.ndarray:
        """Derivative of ReLU"""
        return np.where(x > 0, 1, 0)
    
    def _tanh(self, x: np.ndarray) -> np.ndarray:
        """Tanh activation function (section 5.3)"""
        return np.tanh(x)
    
    def _tanh_derivative(self, x: np.ndarray) -> np.ndarray:
        """Derivative of tanh"""
        return 1 - np.power(np.tanh(x), 2)
    
    # ==================== Forward Propagation ====================
    
    def _forward_pass(self, X: np.ndarray) -> Tuple[List[np.ndarray], List[np.ndarray]]:
        """Forward propagation (section 6)"""
        activations = [X]
        layer_inputs = []
        
        for i in range(self.n_layers - 1):
            layer_input = np.dot(activations[-1], self.weights[i]) + self.biases[i]
            layer_inputs.append(layer_input)
            activation = self.activation_func(layer_input)
            activations.append(activation)
        
        # Output layer (linear for regression)
        last_layer_input = np.dot(activations[-1], self.weights[-1]) + self.biases[-1]
        layer_inputs.append(last_layer_input)
        activations.append(last_layer_input)  # No activation
        
        return activations, layer_inputs
    
    # ==================== Loss Function ====================
    
    def _compute_loss(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        """Compute MSE loss with L2 regularization (section 7)"""
        m = y_true.shape[0]
        mse = np.mean(np.square(y_pred - y_true))
        
        # L2 regularization
        l2_reg = 0
        for w in self.weights:
            l2_reg += np.sum(np.square(w))
        l2_reg *= self.alpha / (2 * m)
        
        return mse + l2_reg
    
    # ==================== Backpropagation ====================
    
    def _backward_pass(
        self, 
        X: np.ndarray, 
        y: np.ndarray, 
        activations: List[np.ndarray], 
        layer_inputs: List[np.ndarray]
    ) -> Tuple[List[np.ndarray], List[np.ndarray]]:
        """Backpropagation (section 8)"""
        m = X.shape[0]
        gradients_w = [None] * self.n_layers
        gradients_b = [None] * self.n_layers
        
        # Gradient of output layer (MSE derivative)
        delta = (activations[-1] - y) * (2 / m)
        
        for i in range(self.n_layers - 1, -1, -1):
            gradients_w[i] = np.dot(activations[i].T, delta) + self.alpha * self.weights[i]
            gradients_b[i] = np.mean(delta, axis=0)
            
            if i > 0:
                delta = np.dot(delta, self.weights[i].T)
                delta *= self.activation_derivative(layer_inputs[i-1])
        
        return gradients_w, gradients_b
    
    # ==================== Optimizers ====================
    
    def _update_weights_sgd(self, gradients_w: List[np.ndarray], gradients_b: List[np.ndarray]) -> None:
        """SGD optimizer (section 9.1)"""
        for i in range(self.n_layers):
            self.weights[i] -= self.learning_rate * gradients_w[i]
            self.biases[i] -= self.learning_rate * gradients_b[i]
    
    def _update_weights_momentum(self, gradients_w: List[np.ndarray], gradients_b: List[np.ndarray]) -> None:
        """Momentum optimizer (section 9.2)"""
        for i in range(self.n_layers):
            self.velocity_weights[i] = self.momentum * self.velocity_weights[i] - self.learning_rate * gradients_w[i]
            self.velocity_biases[i] = self.momentum * self.velocity_biases[i] - self.learning_rate * gradients_b[i]
            self.weights[i] += self.velocity_weights[i]
            self.biases[i] += self.velocity_biases[i]
    
    def _update_weights_rmsprop(self, gradients_w: List[np.ndarray], gradients_b: List[np.ndarray]) -> None:
        """RMSProp optimizer (section 9.3)"""
        decay_rate = 0.9
        for i in range(self.n_layers):
            self.v_weights[i] = decay_rate * self.v_weights[i] + (1 - decay_rate) * np.square(gradients_w[i])
            self.v_biases[i] = decay_rate * self.v_biases[i] + (1 - decay_rate) * np.square(gradients_b[i])
            self.weights[i] -= self.learning_rate * gradients_w[i] / (np.sqrt(self.v_weights[i] + self.epsilon))
            self.biases[i] -= self.learning_rate * gradients_b[i] / (np.sqrt(self.v_biases[i] + self.epsilon))
    
    def _update_weights_adam(self, gradients_w: List[np.ndarray], gradients_b: List[np.ndarray]) -> None:
        """Adam optimizer (section 9.4)"""
        for i in range(self.n_layers):
            self.m_weights[i] = self.beta1 * self.m_weights[i] + (1 - self.beta1) * gradients_w[i]
            self.m_biases[i] = self.beta1 * self.m_biases[i] + (1 - self.beta1) * gradients_b[i]
            self.v_weights[i] = self.beta2 * self.v_weights[i] + (1 - self.beta2) * np.square(gradients_w[i])
            self.v_biases[i] = self.beta2 * self.v_biases[i] + (1 - self.beta2) * np.square(gradients_b[i])
            
            m_w_corr = self.m_weights[i] / (1 - self.beta1 ** self.t)
            m_b_corr = self.m_biases[i] / (1 - self.beta1 ** self.t)
            v_w_corr = self.v_weights[i] / (1 - self.beta2 ** self.t)
            v_b_corr = self.v_biases[i] / (1 - self.beta2 ** self.t)
            
            self.weights[i] -= self.learning_rate * m_w_corr / (np.sqrt(v_w_corr + self.epsilon))
            self.biases[i] -= self.learning_rate * m_b_corr / (np.sqrt(v_b_corr + self.epsilon))
        
        self.t += 1
    
    # ==================== Training ====================
    
    def fit(self, X: np.ndarray, y: np.ndarray) -> 'MLPRegressor':
        """Train the MLP (sections 6-10)"""
        X = np.array(X, dtype=float)
        y_orig = np.array(y, dtype=float)
        
        y = y_orig.reshape(-1, 1) if y_orig.ndim == 1 else y_orig
        
        n_samples, n_features = X.shape
        _, n_outputs = y.shape
        
        self._initialize_weights(n_features, n_outputs)
        
        if self.early_stopping:
            X_train, X_val, y_train, y_val = self._split_train_validation(X, y)
        else:
            X_train, y_train = X, y
        
        update_methods = {
            'sgd': self._update_weights_sgd,
            'momentum': self._update_weights_momentum,
            'rmsprop': self._update_weights_rmsprop,
            'adam': self._update_weights_adam
        }
        update_weights = update_methods[self.solver]
        
        self.loss_history = []
        self.val_loss_history = []
        self.best_loss = np.inf
        self.no_improvement_count = 0
        
        for epoch in range(self.max_iter):
            if self.shuffle:
                indices = np.random.permutation(len(y_train))
                X_train = X_train[indices]
                y_train = y_train[indices]
            
            batch_losses = []
            for i in range(0, len(y_train), self.batch_size):
                X_batch = X_train[i:i+self.batch_size]
                y_batch = y_train[i:i+self.batch_size]
                
                activations, layer_inputs = self._forward_pass(X_batch)
                loss = self._compute_loss(y_batch, activations[-1])
                batch_losses.append(loss)
                
                gradients_w, gradients_b = self._backward_pass(X_batch, y_batch, activations, layer_inputs)
                update_weights(gradients_w, gradients_b)
            
            epoch_loss = np.mean(batch_losses)
            self.loss_history.append(epoch_loss)
            
            if self.early_stopping:
                val_activations, _ = self._forward_pass(X_val)
                val_loss = self._compute_loss(y_val, val_activations[-1])
                self.val_loss_history.append(val_loss)
                
                if val_loss < self.best_loss - self.tol:
                    self.best_loss = val_loss
                    self.no_improvement_count = 0
                else:
                    self.no_improvement_count += 1
                
                if self.no_improvement_count >= self.n_iter_no_change:
                    print(f"Early stopping at epoch {epoch+1}")
                    break
        
        self.trained = True
        return self
    
    def _split_train_validation(self, X: np.ndarray, y: np.ndarray):
        """Split data into training and validation sets"""
        n_samples = X.shape[0]
        n_val = int(n_samples * self.validation_fraction)
        
        if self.shuffle:
            indices = np.random.permutation(n_samples)
            X, y = X[indices], y[indices]
        
        return X[n_val:], X[:n_val], y[n_val:], y[:n_val]
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict target values"""
        if not self.trained:
            raise ValueError("Model must be trained before prediction.")
        
        X = np.array(X, dtype=float)
        activations, _ = self._forward_pass(X)
        y_pred = activations[-1]
        
        return y_pred.ravel() if y_pred.shape[1] == 1 else y_pred
    
    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Return R² coefficient of determination"""
        if not self.trained:
            raise ValueError("Model must be trained before scoring.")
        
        y_true = np.array(y).reshape(-1, 1) if np.array(y).ndim == 1 else np.array(y)
        y_pred = self.predict(X)
        y_pred = y_pred.reshape(-1, 1) if y_pred.ndim == 1 else y_pred
        
        u = ((y_true - y_pred) ** 2).sum()
        v = ((y_true - y_true.mean(axis=0)) ** 2).sum()
        
        return 0.0 if v == 0 else 1 - u / v

## 13. Data Preparation

We generate synthetic regression data with 5 features and 1000 samples.

In [ ]:
# Generate synthetic data
X, y = make_regression(n_samples=1000, n_features=5, noise=0.1, random_state=42)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardization (important for neural networks)
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

print(f"Training set: {X_train_scaled.shape[0]} samples, {X_train_scaled.shape[1]} features")
print(f"Test set: {X_test_scaled.shape[0]} samples")

## 14. Training Our MLPRegressor

In [ ]:
mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=32,
    learning_rate=0.001,
    max_iter=100,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)

mlp.fit(X_train_scaled, y_train_scaled)
print(f"\nTraining completed in {len(mlp.loss_history)} epochs")

## 15. Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(mlp.loss_history, 'b-', linewidth=2, label='Training Loss')
if mlp.early_stopping:
    axes[0].plot(mlp.val_loss_history, 'r-', linewidth=2, label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE + L2)')
axes[0].set_title('Learning Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss on log scale
axes[1].semilogy(mlp.loss_history, 'b-', linewidth=2, label='Training Loss')
if mlp.early_stopping:
    axes[1].semilogy(mlp.val_loss_history, 'r-', linewidth=2, label='Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (log scale)')
axes[1].set_title('Learning Curves (Log Scale)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 16. Model Evaluation

In [ ]:
y_pred_scaled = mlp.predict(X_test_scaled)
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
y_test_original = scaler_y.inverse_transform(y_test_scaled.reshape(-1, 1)).ravel()

mse_custom = mean_squared_error(y_test_original, y_pred)
r2_custom = r2_score(y_test_original, y_pred)

print("=" * 50)
print("Our MLPRegressor Performance")
print("=" * 50)
print(f"Mean Squared Error (MSE): {mse_custom:.4f}")
print(f"R² Score: {r2_custom:.4f}")
print(f"Internal R² Score: {mlp.score(X_test_scaled, y_test_scaled):.4f}")

## 17. Predictions vs True Values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot
axes[0].scatter(y_test_original, y_pred, alpha=0.5, edgecolors='k', linewidth=0.5)
axes[0].plot([y_test_original.min(), y_test_original.max()], 
             [y_test_original.min(), y_test_original.max()], 
             'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('True Values')
axes[0].set_ylabel('Predictions')
axes[0].set_title(f'Predictions vs True Values (R² = {r2_custom:.4f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals
residuals = y_test_original - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, edgecolors='k', linewidth=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predictions')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Residuals - Mean: {np.mean(residuals):.4f}, Std: {np.std(residuals):.4f}")

## 18. Comparison with Scikit-Learn MLPRegressor

In [ ]:
sklearn_mlp = SklearnMLP(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    batch_size=32,
    learning_rate_init=0.001,
    max_iter=100,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    verbose=False
)

sklearn_mlp.fit(X_train_scaled, y_train_scaled)

y_pred_sklearn_scaled = sklearn_mlp.predict(X_test_scaled)
y_pred_sklearn = scaler_y.inverse_transform(y_pred_sklearn_scaled.reshape(-1, 1)).ravel()

mse_sklearn = mean_squared_error(y_test_original, y_pred_sklearn)
r2_sklearn = r2_score(y_test_original, y_pred_sklearn)

print("=" * 60)
print("Performance Comparison")
print("=" * 60)
print(f"{'Metric':<20} {'Our MLP':<18} {'Scikit-Learn':<18}")
print("-" * 60)
print(f"{'MSE':<20} {mse_custom:<18.6f} {mse_sklearn:<18.6f}")
print(f"{'R² Score':<20} {r2_custom:<18.4f} {r2_sklearn:<18.4f}")
print(f"{'Final Loss':<20} {mlp.loss_history[-1]:<18.6f} {sklearn_mlp.loss_curve_[-1]:<18.6f}")
print(f"{'Epochs':<20} {len(mlp.loss_history):<18} {len(sklearn_mlp.loss_curve_):<18}")

## 19. Visual Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Loss curves comparison
axes[0].plot(mlp.loss_history, 'b-', linewidth=2, label='Our MLP')
axes[0].plot(sklearn_mlp.loss_curve_, 'r-', linewidth=2, label='Scikit-Learn')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Predictions comparison
axes[1].scatter(y_test_original, y_pred, alpha=0.5, label='Our MLP', edgecolors='blue')
axes[1].scatter(y_test_original, y_pred_sklearn, alpha=0.5, label='Scikit-Learn', edgecolors='red', marker='x')
axes[1].plot([y_test_original.min(), y_test_original.max()], 
             [y_test_original.min(), y_test_original.max()], 
             'k--', lw=1.5)
axes[1].set_xlabel('True Values')
axes[1].set_ylabel('Predictions')
axes[1].set_title('Predictions Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Bar plot comparison
models = ['Our MLP', 'Scikit-Learn']
r2_scores = [r2_custom, r2_sklearn]
colors = ['#2E86AB', '#A23B72']
bars = axes[2].bar(models, r2_scores, color=colors, edgecolor='black')
axes[2].set_ylim(0, 1.05)
axes[2].set_ylabel('R² Score')
axes[2].set_title('R² Score Comparison')
axes[2].grid(axis='y', alpha=0.3)

for bar, score in zip(bars, r2_scores):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                 f'{score:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 20. Optimizer Comparison

In [ ]:
optimizers = ['sgd', 'momentum', 'rmsprop', 'adam']
optimizer_results = {}

for opt in optimizers:
    print(f"Training with {opt} optimizer...")
    mlp_temp = MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver=opt,
        learning_rate=0.001,
        max_iter=100,
        random_state=42,
        early_stopping=False
    )
    mlp_temp.fit(X_train_scaled, y_train_scaled)
    optimizer_results[opt] = mlp_temp.loss_history

plt.figure(figsize=(12, 6))
colors_opt = {'sgd': 'gray', 'momentum': 'orange', 'rmsprop': 'green', 'adam': 'blue'}
for opt, loss in optimizer_results.items():
    plt.plot(loss, label=opt.upper(), linewidth=2, color=colors_opt[opt])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Optimizer Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nFinal loss after 100 epochs:")
for opt, loss in optimizer_results.items():
    print(f"  {opt.upper()}: {loss[-1]:.6f}")

## 21. Activation Function Comparison

In [ ]:
activations = ['relu', 'tanh', 'sigmoid', 'leaky_relu']
activation_results = {}

for act in activations:
    print(f"Training with {act} activation...")
    mlp_temp = MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation=act,
        solver="adam",
        learning_rate=0.001,
        max_iter=100,
        random_state=42,
        early_stopping=False
    )
    mlp_temp.fit(X_train_scaled, y_train_scaled)
    activation_results[act] = mlp_temp.loss_history

plt.figure(figsize=(12, 6))
colors_act = {'relu': 'blue', 'tanh': 'red', 'sigmoid': 'green', 'leaky_relu': 'orange'}
for act, loss in activation_results.items():
    plt.plot(loss, label=act.upper(), linewidth=2, color=colors_act[act])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Activation Function Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nFinal loss after 100 epochs:")
for act, loss in activation_results.items():
    print(f"  {act.upper()}: {loss[-1]:.6f}")

## 22. Conclusion

### Key Takeaways

1. **MLP for regression** works well for non-linear problems
2. **Activation functions** matter: ReLU and Leaky ReLU generally outperform sigmoid and tanh
3. **Optimizers** comparison: Adam and RMSProp converge faster than SGD
4. **Our implementation** achieves comparable performance to scikit-learn
5. **Early stopping** helps prevent overfitting

### Future Improvements

- Add batch normalization
- Implement learning rate scheduling
- Add dropout for regularization
- Support for GPU acceleration

### References

- Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986). Learning representations by back-propagating errors.
- Kingma, D. P., & Ba, J. (2014). Adam: A Method for Stochastic Optimization.
- Glorot, X., & Bengio, Y. (2010). Understanding the difficulty of training deep feedforward neural networks.